# maplib on Databricks: Knowledge Graphs in Your Lakehouse

Read from Delta tables, build a knowledge graph, run SPARQL, validate with SHACL — all in one notebook.

> This notebook is designed for Databricks. The `spark` object is available by default in Databricks notebooks.

---
## Read from Delta Lake

In [ ]:
import polars as pl
from maplib import Model

# Option A: Spark -> Polars via Arrow (large tables, push filters to Spark first)
# equipment_spark = spark.read.table("manufacturing.equipment").filter("site = 'Oslo'")
# df_equip = pl.from_pandas(equipment_spark.toPandas())

# Option B: Direct Polars read (small-medium Delta tables)
# df_equip = pl.read_delta("dbfs:/mnt/data/equipment/")

# For this demo, we use inline sample data:
df_sites = pl.DataFrame({
    "site":   ["Oslo","Bergen","Stavanger","Tromsø"],
    "region": ["East","West","West","North"],
})

df_equip = pl.DataFrame({
    "equipment_id": ["EQ-001","EQ-002","EQ-003","EQ-004","EQ-005","EQ-006","EQ-007","EQ-008"],
    "name":         ["Compressor Alpha","Pump Beta","Turbine Gamma","Heat Exchanger Delta",
                     "Valve Epsilon","Compressor Zeta","Pump Eta","Turbine Theta"],
    "site":         ["Oslo","Oslo","Bergen","Bergen","Stavanger","Stavanger","Tromsø","Tromsø"],
    "equip_type":   ["Compressor","Pump","Turbine","HeatExchanger",
                     "Valve","Compressor","Pump","Turbine"],
    "install_year": [2008, 2015, 2003, 2019, 2011, 2017, 2006, 2021],
})

df_workorders = pl.DataFrame({
    "wo_id":        ["WO-101","WO-102","WO-103","WO-104","WO-105","WO-106","WO-107","WO-108","WO-109","WO-110"],
    "equipment_id": ["EQ-001","EQ-003","EQ-001","EQ-005","EQ-003","EQ-007","EQ-002","EQ-006","EQ-008","EQ-004"],
    "priority":     ["critical","normal","normal","critical","critical","critical","normal","normal","critical","normal"],
    "category":     ["vibration","leak","electrical","corrosion","vibration",
                     "vibration","seal","electrical","alignment","fouling"],
    "cost_usd":     [12500, 3200, 8700, 15600, 22000, 9800, 4100, 6500, 11200, 7300],
    "date":         ["2024-11-15","2024-10-03","2025-01-22","2024-09-18","2025-03-05",
                     "2025-02-14","2024-12-01","2025-04-10","2025-05-20","2024-08-30"],
})

df_readings = pl.DataFrame({
    "equipment_id": ["EQ-001","EQ-001","EQ-002","EQ-003","EQ-003","EQ-004","EQ-005","EQ-006","EQ-007","EQ-008"],
    "metric":       ["temperature","vibration","pressure","temperature","vibration",
                     "temperature","pressure","temperature","pressure","temperature"],
    "value":        [87.3, 4.2, 14.7, 95.1, 6.8, 52.4, 11.9, 68.5, 16.3, 71.2],
    "status":       ["warning","alarm","normal","alarm","alarm","normal","normal","normal","warning","normal"],
})

print(f"Sites: {df_sites.shape[0]} rows")
print(f"Equipment: {df_equip.shape[0]} rows")
print(f"Work orders: {df_workorders.shape[0]} rows")
print(f"Sensor readings: {df_readings.shape[0]} rows")

---
## Build the knowledge graph

Four DataFrames, four OTTR templates, one graph. The templates are where the join keys become real relationships: `mfg:locatedAt` points equipment at a site IRI, and the `mfg:Site` template gives that IRI a label — without it every query that joins through a site comes back empty.

In [ ]:
m = Model()
mfg = "http://example.com/manufacturing/"

m.add_template("""
    @prefix mfg:  <http://example.com/manufacturing/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

    mfg:Site [
        ottr:IRI    ?site,
        xsd:string  ?name,
        xsd:string  ?region
    ] :: {
        ottr:Triple(?site, a,          mfg:Site),
        ottr:Triple(?site, rdfs:label, ?name),
        ottr:Triple(?site, mfg:region, ?region)
    } .

    mfg:Equipment [
        ottr:IRI    ?equip,
        xsd:string  ?name,
        ottr:IRI    ?site,
        ottr:IRI    ?equip_class,
        xsd:integer ?install_year
    ] :: {
        ottr:Triple(?equip, a,                  ?equip_class),
        ottr:Triple(?equip, rdfs:label,         ?name),
        ottr:Triple(?equip, mfg:locatedAt,      ?site),
        ottr:Triple(?equip, mfg:installedYear,  ?install_year)
    } .

    mfg:WorkOrder [
        ottr:IRI    ?wo,
        ottr:IRI    ?equip,
        xsd:string  ?priority,
        xsd:string  ?category,
        xsd:double  ?cost,
        xsd:date    ?date
    ] :: {
        ottr:Triple(?wo, a,                mfg:WorkOrder),
        ottr:Triple(?wo, mfg:forEquipment, ?equip),
        ottr:Triple(?wo, mfg:priority,     ?priority),
        ottr:Triple(?wo, mfg:category,     ?category),
        ottr:Triple(?wo, mfg:cost,         ?cost),
        ottr:Triple(?wo, mfg:date,         ?date)
    } .

    mfg:Reading [
        ottr:IRI    ?reading,
        ottr:IRI    ?equip,
        xsd:string  ?metric,
        xsd:double  ?value,
        ottr:IRI    ?status
    ] :: {
        ottr:Triple(?reading, a,             mfg:Reading),
        ottr:Triple(?equip,   mfg:hasReading, ?reading),
        ottr:Triple(?reading, mfg:metric,    ?metric),
        ottr:Triple(?reading, mfg:value,     ?value),
        ottr:Triple(?reading, mfg:status,    ?status)
    } .
""")

In [ ]:
# Map all four sources
m.map(mfg + "Site", df_sites.select(
    site   = pl.lit(mfg + "site/") + pl.col("site"),
    name   = "site",
    region = "region",
))

m.map(mfg + "Equipment", df_equip.select(
    equip       = pl.lit(mfg + "equip/") + pl.col("equipment_id"),
    name        = "name",
    site        = pl.lit(mfg + "site/") + pl.col("site"),
    equip_class = pl.lit(mfg) + pl.col("equip_type"),
    install_year = "install_year",
))

m.map(mfg + "WorkOrder", df_workorders.select(
    wo       = pl.lit(mfg + "wo/") + pl.col("wo_id"),
    equip    = pl.lit(mfg + "equip/") + pl.col("equipment_id"),
    priority = "priority",
    category = "category",
    cost     = pl.col("cost_usd").cast(pl.Float64),   # template declares xsd:double
    date     = pl.col("date").str.to_date(),
))

m.map(mfg + "Reading", df_readings.select(
    # one equipment can have several metrics, so the reading IRI needs both
    reading = pl.lit(mfg + "reading/") + pl.col("equipment_id") + pl.lit("-") + pl.col("metric"),
    equip   = pl.lit(mfg + "equip/") + pl.col("equipment_id"),
    metric  = "metric",
    value   = "value",
    status  = pl.lit(mfg) + pl.col("status"),
))

print(f"Knowledge graph: {m.size()} triples from 4 sources")

---
## SPARQL on lakehouse data

### Site scorecard — where is the maintenance budget going?

One query spans the site, equipment and work-order tables. `IF` inside `SUM` gives a conditional count without a second pass.

In [ ]:
m.query("""
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?site ?region
           (COUNT(DISTINCT ?eq) AS ?assets)
           (COUNT(?wo)          AS ?work_orders)
           (SUM(IF(?priority = "critical", 1, 0)) AS ?critical)
           (SUM(?cost)          AS ?total_cost)
    WHERE {
        ?s  rdfs:label       ?site ;
            mfg:region       ?region .
        ?eq mfg:locatedAt    ?s .
        ?wo mfg:forEquipment ?eq ;
            mfg:priority     ?priority ;
            mfg:cost         ?cost .
    }
    GROUP BY ?site ?region
    ORDER BY DESC(?total_cost)
""")

### Cross-source insight: old equipment with sensor alarms

Equipment table + sensor readings + work orders — three sources, one query.

In [ ]:
m.query("""
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?name ?site ?install_year ?metric ?value ?n_workorders ?lifetime_spend
    WHERE {
        ?eq rdfs:label        ?name ;
            mfg:locatedAt     ?s ;
            mfg:installedYear ?install_year ;
            mfg:hasReading    ?r .
        ?s  rdfs:label        ?site .
        ?r  mfg:status        mfg:alarm ;
            mfg:metric        ?metric ;
            mfg:value         ?value .
        {
            SELECT ?eq (COUNT(?wo) AS ?n_workorders) (SUM(?cost) AS ?lifetime_spend)
            WHERE { ?wo mfg:forEquipment ?eq ; mfg:cost ?cost }
            GROUP BY ?eq
        }
        FILTER(?install_year < 2010)
    }
    ORDER BY DESC(?lifetime_spend)
""")

### Maintenance cost by failure category and site

In [ ]:
m.query("""
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?site ?category
           (COUNT(?wo)  AS ?incidents)
           (SUM(?cost)  AS ?total_cost)
    WHERE {
        ?eq mfg:locatedAt     ?s .
        ?s  rdfs:label        ?site .
        ?wo mfg:forEquipment  ?eq ;
            mfg:category      ?category ;
            mfg:cost          ?cost .
    }
    GROUP BY ?site ?category
    ORDER BY ?site DESC(?total_cost)
""")

---
## CONSTRUCT: turn an analysis into new graph structure

`SELECT` hands you a table and the reasoning is gone. `CONSTRUCT` hands you *triples* — the conclusion becomes part of the graph, so the next query can build on it.

Here we define what a high-risk asset is once, in one query: pre-2010, currently in alarm, and expensive to keep running.

In [ ]:
RISK_RULE = """
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT {
        ?eq a                  mfg:HighRiskAsset ;
            mfg:riskReason     ?reason ;
            mfg:lifetimeSpend  ?spend ;
            mfg:alarmingMetric ?metric .
    }
    WHERE {
        ?eq mfg:installedYear ?install_year ;
            mfg:hasReading    ?r .
        ?r  mfg:status        mfg:alarm ;
            mfg:metric        ?metric .
        {
            SELECT ?eq (SUM(?cost) AS ?spend)
            WHERE { ?wo mfg:forEquipment ?eq ; mfg:cost ?cost }
            GROUP BY ?eq
        }
        FILTER(?install_year < 2010 && ?spend > 20000)
        BIND(CONCAT("installed ", STR(?install_year),
                    ", lifetime spend ", STR(?spend)) AS ?reason)
    }
"""

# Preview the triples the rule would produce — nothing is written to the graph yet.
# maplib returns one DataFrame per (predicate, object-type) group.
for df in m.query(RISK_RULE):
    print(df)

`m.insert()` runs the same CONSTRUCT and writes the result back into the graph. `mfg:HighRiskAsset` now exists as a class nothing in the source tables ever mentioned.

In [ ]:
before = m.size()
m.insert(RISK_RULE)
print(f"{m.size() - before} derived triples added ({before} -> {m.size()})")

# Downstream queries just ask for the class — the risk definition lives in one place
m.query("""
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?asset ?site ?region ?lifetime_spend
           (COUNT(DISTINCT ?metric) AS ?metrics_in_alarm)
           ?why
    WHERE {
        ?eq a                  mfg:HighRiskAsset ;
            rdfs:label         ?asset ;
            mfg:lifetimeSpend  ?lifetime_spend ;
            mfg:riskReason     ?why ;
            mfg:alarmingMetric ?metric ;
            mfg:locatedAt      ?s .
        ?s  rdfs:label         ?site ;
            mfg:region         ?region .
    }
    GROUP BY ?asset ?site ?region ?lifetime_spend ?why
    ORDER BY DESC(?lifetime_spend)
""")

---
## Write results back to Delta

Close the loop: graph insights -> Delta table -> BI dashboards.

In [ ]:
risk_df = m.query("""
    PREFIX mfg:  <http://example.com/manufacturing/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?equipment ?site ?region ?install_year
           (COUNT(?wo) AS ?work_orders)
           (SUM(IF(?priority = "critical", 1, 0)) AS ?critical_wos)
           (SUM(?cost) AS ?total_cost)
           (IF(BOUND(?flag), "HIGH", "normal") AS ?risk_tier)
    WHERE {
        ?eq rdfs:label        ?equipment ;
            mfg:locatedAt     ?s ;
            mfg:installedYear ?install_year .
        ?s  rdfs:label        ?site ;
            mfg:region        ?region .
        ?wo mfg:forEquipment  ?eq ;
            mfg:priority      ?priority ;
            mfg:cost          ?cost .
        # the tier comes from the triples CONSTRUCT wrote back above
        OPTIONAL { ?eq a ?flag . FILTER(?flag = mfg:HighRiskAsset) }
    }
    GROUP BY ?equipment ?site ?region ?install_year ?flag
    ORDER BY DESC(?total_cost)
""")

# On Databricks:
# spark.createDataFrame(risk_df.to_pandas()).write.format("delta").mode("overwrite").saveAsTable("analytics.equipment_risk")

print("Equipment risk report:")
risk_df